# Ant Mutilated – PPO Checkpoint Replay

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["MUJOCO_GL"] = "egl"

import json
from pathlib import Path

import jax
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
import mujoco
import mediapy as media
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mujoco import mjx
from mujoco_playground._src.dm_control_suite import common
from brax.training import checkpoint as brax_checkpoint
from brax.training import networks as brax_networks
from brax.training.acme import running_statistics
from brax.training.agents.ppo import networks as ppo_networks

import crawler_playground
from crawler_playground.envs.ant.ant import Run as AntRun, default_config
from crawler_playground.envs.ant.randomize import VALID_VARIANTS

print("JAX devices:", jax.devices())

## Configuration

In [ ]:
# ── Training runs to compare ────────────────────────────────────────────────────
RUNS = [
    {"dir": "/home/alexanderdittrich/Downloads/tamara/ant-mutilated-ppo-260318214550", "name": "5"},
    {"dir": "/home/alexanderdittrich/Downloads/tamara/ant-mutilated-ppo-260318211137", "name": "4"},
    {"dir": "/home/alexanderdittrich/Downloads/tamara/ant-mutilated-ppo-260318203733", "name": "3"},
    {"dir": "/home/alexanderdittrich/Downloads/tamara/ant-mutilated-ppo-260318200329", "name": "2"},
    {"dir": "/home/alexanderdittrich/Downloads/tamara/ant-mutilated-ppo-260318184137", "name": "1"},
]

TRAIN_VARIANTS = ["none", "FR", "FL"]
TEST_VARIANTS = ["RR", "RL"]
EVAL_VARIANTS = ["none", "FR", "FL", "RL", "RR"]
NUM_SEEDS = 30
EPISODE_LENGTH = 500  # max steps per rollout
# ──────────────────────────────────────────────────────────────────────────────

## Checkpoint loader

In [ ]:
import inspect

_KERNEL_INIT_KEYS = (
    "policy_network_kernel_init_fn",
    "value_network_kernel_init_fn",
    "q_network_kernel_init_fn",
    "mean_kernel_init_fn",
)

_PPO_NETWORK_VALID_KEYS = set(
    inspect.signature(ppo_networks.make_ppo_networks).parameters.keys()
)


def load_inference_fn(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    step_dirs = sorted(
        [
            d
            for d in checkpoint_dir.iterdir()
            if d.is_dir() and d.name.isdigit()
        ],
        key=lambda d: int(d.name),
    )
    if not step_dirs:
        raise FileNotFoundError(
            f"No checkpoint directories found in {checkpoint_dir}"
        )
    step = int(step_dirs[-1].name)
    ckpt_path = checkpoint_dir / f"{step:012d}"

    raw_params = brax_checkpoint.load(ckpt_path)
    normalizer_params, policy_params, _ = raw_params

    with open(ckpt_path / "ppo_network_config.json") as f:
        cfg_raw = json.load(f)

    kw = {k: v for k, v in cfg_raw["network_factory_kwargs"].items()
          if k in _PPO_NETWORK_VALID_KEYS}
    if "activation" in kw and isinstance(kw["activation"], str):
        kw["activation"] = brax_networks.ACTIVATION[kw["activation"]]
    for key in _KERNEL_INIT_KEYS:
        if key in kw and kw[key] is not None:
            kw[key] = brax_networks.KERNEL_INITIALIZER[kw[key]]

    obs_raw = cfg_raw["observation_size"]
    observation_size = (
        int(obs_raw["shape"][0]) if isinstance(obs_raw, dict) else int(obs_raw)
    )
    normalize_observations = cfg_raw.get("normalize_observations", False)
    preprocess_fn = (
        running_statistics.normalize
        if normalize_observations
        else (lambda x, y: x)
    )

    ppo_network = ppo_networks.make_ppo_networks(
        observation_size=observation_size,
        action_size=cfg_raw["action_size"],
        preprocess_observations_fn=preprocess_fn,
        **kw,
    )
    make_policy_ = ppo_networks.make_inference_fn(ppo_network)
    params = (normalizer_params, policy_params)
    inference_fn = jax.jit(make_policy_(params, deterministic=True))

    return inference_fn, step, make_policy_, params

## Environment factory

In [ ]:
_ANT_XML_DIR = (
    Path(crawler_playground.__file__).parent / "envs" / "ant" / "xmls"
)


def make_variant_env(variant: str) -> AntRun:
    """Return an AntRun with the correct XML for *variant* (visual + physical)."""
    assert variant in VALID_VARIANTS, f"Unknown variant '{variant}'"
    env = AntRun(config=default_config())
    if variant != "none":
        xml_path = _ANT_XML_DIR / f"ant_mutilated_{variant}.xml"
        mj_model = mujoco.MjModel.from_xml_string(
            xml_path.read_text(), common.get_assets()
        )
        mj_model.opt.timestep = env.sim_dt
        env._mj_model = mj_model
        env._mjx_model = mjx.put_model(mj_model)
    return env

## Rollout helpers

In [ ]:
import jax.numpy as jnp

# ── Sequential rollout (returns state history — for visualisation) ─────────────


def run_rollout(env: AntRun, inference_fn, seed: int = 0):
    """Run one episode; return (state_history, action_history, total_reward)."""
    env_reset = jax.jit(env.reset)
    env_step = jax.jit(env.step)

    rng = jax.random.key(seed)
    rng, rng_reset = jax.random.split(rng)
    state = env_reset(rng_reset)

    state_history, action_history, total_reward = [], [], 0.0
    for _ in range(EPISODE_LENGTH):
        rng, rng_act = jax.random.split(rng)
        actions, _ = inference_fn(state.obs, rng_act)
        state = env_step(state, actions)
        state_history.append(state)
        action_history.append(np.asarray(actions))
        total_reward += float(state.reward)
        if float(state.done) > 0.5:
            break
    return state_history, action_history, total_reward


# ── Batched rollout (vmap over seeds, lax.scan over steps) ────────────────────


def make_batched_rollout(env, inference_fn):
    _reset = env.reset
    _step = env.step

    def single(seed):
        rng = jax.random.key(seed)
        rng, rng_reset = jax.random.split(rng)
        state = _reset(rng_reset)

        def scan_step(carry, _):
            state, rng, total, active = carry
            rng, rng_act = jax.random.split(rng)
            actions, _ = inference_fn(state.obs, rng_act)
            new_state = _step(state, actions)
            total = total + new_state.reward * active
            active = active * (1.0 - new_state.done)
            return (new_state, rng, total, active), None

        (_, _, total_reward, _), _ = jax.lax.scan(
            scan_step,
            (state, rng, jnp.zeros(()), jnp.ones(())),
            None,
            length=EPISODE_LENGTH,
        )
        return total_reward

    return jax.jit(jax.vmap(single))

## Evaluation

In [ ]:
per_run = {} 
pooled = {v: [] for v in EVAL_VARIANTS}
run_meta = {}

seed_batch = jnp.arange(NUM_SEEDS) 

for run in RUNS:
    run_name = run["name"]
    print(f"\n{'─' * 55}")
    print(f"  {run_name}  ({run['dir']})")

    infer_fn, last_step, _, _ = load_inference_fn(run["dir"])
    run_meta[run_name] = {"step": last_step}
    print(f"  Last checkpoint: step {last_step:,}")

    run_results = {}
    for variant in EVAL_VARIANTS:
        env = make_variant_env(variant)
        batched_fn = make_batched_rollout(env, infer_fn)
        rewards = np.array(batched_fn(seed_batch))

        run_results[variant] = rewards
        pooled[variant].extend(rewards)

        mean = rewards.mean()
        sem = rewards.std(ddof=1) / np.sqrt(len(rewards))
        split = "train" if variant in TRAIN_VARIANTS else "test "
        print(
            f"  [{split}] {variant:4s}  {mean:7.1f} ± {sem:5.1f} SEM"
            f"  (min={rewards.min():.0f}  max={rewards.max():.0f})"
        )

    per_run[run_name] = run_results

for v in EVAL_VARIANTS:
    pooled[v] = np.array(pooled[v])

n_total = len(RUNS) * NUM_SEEDS
print(f"\n{'─' * 55}")
print(
    f"Pooled n per variant: {n_total}  ({len(RUNS)} runs × {NUM_SEEDS} seeds)"
)

## Reward comparison — mean ± SEM bar chart

In [ ]:
n_total = len(RUNS) * NUM_SEEDS
t_crit = scipy_stats.t.ppf(0.975, df=n_total - 1)

fig, ax = plt.subplots(figsize=(max(5, 1.6 * len(EVAL_VARIANTS)), 5))

for vi, variant in enumerate(EVAL_VARIANTS):
    rewards = pooled[variant]
    mean = rewards.mean()
    sem = rewards.std(ddof=1) / np.sqrt(n_total)
    color = "steelblue" if variant in TRAIN_VARIANTS else "coral"

    ax.bar(vi, mean, color=color, alpha=0.80, zorder=2)
    ax.errorbar(
        vi,
        mean,
        yerr=sem,
        fmt="none",
        color="black",
        capsize=4,
        linewidth=1.5,
        zorder=3,
    )

    # Per-run means as dots (shows between-run variability)
    run_means = [per_run[r["name"]][variant].mean() for r in RUNS]
    ax.scatter(
        [vi] * len(run_means),
        run_means,
        color="black",
        s=22,
        zorder=4,
        alpha=0.65,
        label="run mean" if vi == 0 else "",
    )

# Divider
if TRAIN_VARIANTS and TEST_VARIANTS:
    div = len(TRAIN_VARIANTS) - 0.5
    ax.axvline(div, color="dimgray", linestyle="--", linewidth=0.9)
    ylo, yhi = ax.get_ylim()
    ax.text(
        div + 0.06,
        ylo + (yhi - ylo) * 0.01,
        "← train  |  test →",
        fontsize=7.5,
        color="dimgray",
        va="bottom",
    )

ax.axhline(0, color="black", linewidth=0.5, linestyle=":")
ax.set_xticks(range(len(EVAL_VARIANTS)))
ax.set_xticklabels(EVAL_VARIANTS, fontsize=11)
ax.set_ylabel("Total reward  (mean ± SEM)", fontsize=11)
ax.set_title(
    f"Aggregate performance  ({len(RUNS)} runs × {NUM_SEEDS} seeds = n={n_total} per variant)\n"
    f"95 % CI = mean ± {t_crit:.3f}×SEM,  df={n_total - 1}",
    fontsize=9,
)
ax.legend(
    handles=[
        mpatches.Patch(color="steelblue", label="Train variant"),
        mpatches.Patch(color="coral", label="Test variant"),
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor="black",
            markersize=6,
            label=f"Per-run mean  (n={NUM_SEEDS})",
        ),
    ],
    fontsize=8,
)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Statistics table  (mean ± SEM  and  95 % CI)

In [ ]:
rows = []
for variant in EVAL_VARIANTS:
    rewards = pooled[variant]
    n_ = len(rewards)
    mean = rewards.mean()
    sem = rewards.std(ddof=1) / np.sqrt(n_)
    ci_lo = mean - t_crit * sem
    ci_hi = mean + t_crit * sem
    run_means = [per_run[r["name"]][variant].mean() for r in RUNS]
    rows.append(
        {
            "Variant": variant,
            "Split": "train" if variant in TRAIN_VARIANTS else "test",
            "n": n_,
            "Mean ± SEM": f"{mean:.1f} ± {sem:.1f}",
            f"95% CI (t={t_crit:.2f})": f"[{ci_lo:.1f},  {ci_hi:.1f}]",
            "Per-run means": "  ".join(f"{m:.0f}" for m in run_means),
        }
    )

df = pd.DataFrame(rows).set_index("Variant")
pd.set_option("display.max_rows", None)
display(df)

## Single-run visualisation

In [ ]:
VIZ_RUN = 0
VIZ_SEEDS = 4
RENDER_VARIANT = "none"  # variant to render as video; None → skip

viz_run_name = RUNS[VIZ_RUN]["name"]
viz_infer_fn, viz_step, viz_make_policy, viz_params = load_inference_fn(
    RUNS[VIZ_RUN]["dir"]
)

viz_results = {}
for variant in EVAL_VARIANTS:
    env = make_variant_env(variant)
    histories = [
        run_rollout(env, viz_infer_fn, seed=s) for s in range(VIZ_SEEDS)
    ]
    viz_results[variant] = {
        "env": env,
        "rewards": [h[2] for h in histories],
        "histories": [(h[0], h[1]) for h in histories],
    }

print(f"Visualising run: {viz_run_name}  (step {viz_step:,})")

In [ ]:
fig, axs = plt.subplots(
    1,
    len(EVAL_VARIANTS),
    figsize=(3.5 * len(EVAL_VARIANTS), 4),
    sharey=True,
)
if len(EVAL_VARIANTS) == 1:
    axs = [axs]

for ax, variant in zip(axs, EVAL_VARIANTS):
    for state_history, _ in viz_results[variant]["histories"]:
        xpos = np.array(
            [s.data.xpos[1] for s in state_history]
        )  # torso = body 1
        ax.plot(xpos[:, 0], xpos[:, 1], alpha=0.7, linewidth=1.2)
        ax.scatter(xpos[0, 0], xpos[0, 1], s=30, color="green", zorder=5)
        ax.scatter(xpos[-1, 0], xpos[-1, 1], s=30, color="red", zorder=5)

    color = "steelblue" if variant in TRAIN_VARIANTS else "coral"
    split = "train" if variant in TRAIN_VARIANTS else "test"
    ax.set_title(f"{variant}  ({split})", color=color)
    ax.set_xlabel("x (m)")
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)

axs[0].set_ylabel("y (m)")
plt.suptitle(
    f"Top-down trajectories  (● start  ● end)   [{viz_run_name}  step {viz_step:,}]",
    y=1.02,
)
plt.tight_layout()
plt.show()

## Forward velocity over time

In [ ]:
_dt = make_variant_env("none").dt  # ctrl_dt

fig, axs = plt.subplots(
    1,
    len(EVAL_VARIANTS),
    figsize=(3.5 * len(EVAL_VARIANTS), 3),
    sharey=True,
)
if len(EVAL_VARIANTS) == 1:
    axs = [axs]

for ax, variant in zip(axs, EVAL_VARIANTS):
    for state_history, _ in viz_results[variant]["histories"]:
        xpos = np.array([s.data.xpos[1, 0] for s in state_history])
        velocity = np.diff(xpos) / _dt
        ax.plot(velocity, alpha=0.7, linewidth=1)
    ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
    ax.set_title(variant)
    ax.set_xlabel("Step")
    ax.grid(alpha=0.3)

axs[0].set_ylabel("Forward velocity (m/s)")
plt.suptitle(f"Forward velocity  [{viz_run_name}  step {viz_step:,}]")
plt.tight_layout()
plt.show()

## Video rendering

In [ ]:
render_variants = EVAL_VARIANTS if RENDER_VARIANT is None else [RENDER_VARIANT]

scene_option = mujoco.MjvOption()
scene_option.geomgroup[2] = True
scene_option.geomgroup[3] = False
scene_option.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False

for variant in render_variants:
    env = viz_results[variant]["env"]
    state_history = viz_results[variant]["histories"][0][0]
    reward = viz_results[variant]["rewards"][0]

    render_every = 1
    fps = (1.0 / env.dt) / render_every
    traj = state_history[::render_every]

    frames = env.render(
        traj, height=480, width=640, camera=0, scene_option=scene_option
    )
    print(
        f"[{viz_run_name}] {variant}  |  reward = {reward:.1f}  |  {len(frames)} frames @ {fps:.0f} fps"
    )
    media.show_video(frames, fps=fps)

## Learning curve

Evaluates the *none* variant at every saved step to show training progress.

In [ ]:
CURVE_VARIANT = "none"
CURVE_SEED = 0

viz_checkpoint_dir = Path(RUNS[VIZ_RUN]["dir"])
viz_available_steps = sorted(
    int(d.name)
    for d in viz_checkpoint_dir.iterdir()
    if d.is_dir() and d.name.isdigit()
)

env_curve = make_variant_env(CURVE_VARIANT)
curve_steps, curve_rewards = [], []

for s in viz_available_steps:
    ckpt = viz_checkpoint_dir / f"{s:012d}"
    raw = brax_checkpoint.load(ckpt)

    with open(ckpt / "ppo_network_config.json") as f:
        cfg_s = json.load(f)
    kw_s = {k: v for k, v in cfg_s["network_factory_kwargs"].items()
            if k in _PPO_NETWORK_VALID_KEYS}
    if "activation" in kw_s and isinstance(kw_s["activation"], str):
        kw_s["activation"] = brax_networks.ACTIVATION[kw_s["activation"]]
    for key in _KERNEL_INIT_KEYS:
        if key in kw_s and kw_s[key] is not None:
            kw_s[key] = brax_networks.KERNEL_INITIALIZER[kw_s[key]]
    obs_raw_s = cfg_s["observation_size"]
    obs_size_s = (
        int(obs_raw_s["shape"][0])
        if isinstance(obs_raw_s, dict)
        else int(obs_raw_s)
    )
    pre_s = (
        running_statistics.normalize
        if cfg_s.get("normalize_observations")
        else (lambda x, y: x)
    )
    net_s = ppo_networks.make_ppo_networks(
        obs_size_s,
        cfg_s["action_size"],
        preprocess_observations_fn=pre_s,
        **kw_s,
    )
    infer_s = jax.jit(
        ppo_networks.make_inference_fn(net_s)(
            (raw[0], raw[1]), deterministic=True
        )
    )

    _, _, r = run_rollout(env_curve, infer_s, seed=CURVE_SEED)
    curve_steps.append(s)
    curve_rewards.append(r)
    print(f"  step {s:>12,d}  reward = {r:.1f}")

best_idx = int(np.argmax(curve_rewards))
print(
    f"\nBest: step {curve_steps[best_idx]:,}  reward = {curve_rewards[best_idx]:.1f}"
)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(curve_steps, curve_rewards, marker="o", linewidth=1.5)
ax.axvline(
    curve_steps[best_idx],
    color="green",
    linestyle="--",
    linewidth=1,
    label=f"best  step {curve_steps[best_idx]:,}",
)
ax.axhline(0, color="black", linewidth=0.5, linestyle=":")
ax.set_xlabel("Training steps")
ax.set_ylabel("Total reward")
ax.set_title(
    f"Learning curve – {viz_run_name}  variant '{CURVE_VARIANT}'  (seed {CURVE_SEED})"
)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()